# Agregados por año y país (incremental)

Procesa el lote `p_file_date`: suma presupuesto e ingresos por año y país para las películas desde 2015, calcula rankings por año y carga el resultado en `movie_gold.results_group_movie_country` con `MERGE`.

### Configuración

Carga las rutas y las funciones comunes, y lee `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, sum, avg, rank, dense_rank, row_number, lit
from pyspark.sql.window import Window

### Lectura de fuentes Silver

Lee de `movie_silver` las tablas del lote (`file_date = p_file_date`): `movies` y `production_countries`. La dimensión `countries` se lee completa.

In [0]:
movies_df = (
    spark.read
        .table("movie_silver.movies")
        .filter(f"file_date = '{v_file_date}'")
).select("movie_id", "budget", "revenue", "year_release_date")

In [0]:
countries_df = (
    spark.read
        .table("movie_silver.countries")
).select("country_id", "country_name")

In [0]:
production_countries_df = (
    spark.read
        .table("movie_silver.production_countries")
        .filter(col("file_date") == v_file_date)
)


### Agregación y rankings

Une películas desde 2015 con sus países, suma presupuesto e ingresos por año y país, calcula rankings dentro de cada año (`row_number`, `dense_rank`) y agrega `created_date`.

In [0]:
rank_budget_revenue = (
    Window.partitionBy("year_release_date").
        orderBy(
            col("total_budget").desc(), col("total_revenue").desc()
            )
)

results_group_movie_country = (
    movies_df.filter(movies_df.year_release_date >= 2015)
        .join(production_countries_df,
              movies_df["movie_id"] == production_countries_df["movie_id"], "inner")
        .join(countries_df,
              production_countries_df["country_id"] == countries_df["country_id"], "inner")
        .select(
            "year_release_date",
            "country_name",
            "budget",
            "revenue"
        ).groupBy("year_release_date", "country_name")
            .agg(
                sum("budget").alias("total_budget"),
                sum("revenue").alias("total_revenue")
        )
        .orderBy(col("total_budget").desc())
        .withColumn("budget_rank",
                    row_number()
                        .over(Window.partitionBy("year_release_date")
                              .orderBy(col("total_budget").desc())))
        .withColumn("rank_budget_revenue",
                    dense_rank().over(rank_budget_revenue))
        .withColumn("created_date", lit(v_file_date))
)
display(results_group_movie_country)

### Escritura en Gold

Las celdas comentadas conservan la estrategia anterior (borrar partición + `append`). La carga activa usa `MERGE` más abajo.

In [0]:
#overwrite_partitioned_table("movie_silver", "results_group_movie_country","created_date", v_file_date)

In [0]:
"""
(
    results_group_movie_country
        .write
            .mode("append")
            .saveAsTable(f"movie_gold.results_group_movie_country")
)
"""

## Utilizando merge
Carga el resultado con `merge_delta_lake` (llave `year_release_date` + `country_name` + `created_date`), particionando por `created_date`.

In [0]:
merge_condition = 'trgt.year_release_date = src.year_release_date AND trgt.country_name = src.country_name AND trgt.created_date = src.created_date'
merge_delta_lake(input_df = results_group_movie_country, schema_name = "movie_gold", table_name = "results_group_movie_country", merge_condition = merge_condition, partition_column = "created_date")

In [0]:
%sql
SELECT created_date, COUNT(*) FROM movie_gold.results_group_movie_country
GROUP BY created_date
ORDER BY created_date